In [1]:
# download model weights
!curl -L "https://drive.usercontent.google.com/download?id=1dlwaElu0dQQdoEeJkuP2LKGx1TSCjE-z&confirm=xxx" --output yolopx.pth

/bin/bash: curl: command not found


In [ ]:
PATH = "epoch-195.pth"
OUTPUT_PATH = "yolopx_384.onnx"

import torch
from lib.config import cfg
from lib.models import get_net
from lib.utils.utils import select_device
import os 
a= os.getcwd()

# load model
logger = None
device = select_device(logger, '0')

model = get_net(cfg)
checkpoint = torch.load(PATH, map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)

/tmp/ipykernel_114852/3127850572.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH, map_location=device)


In [2]:
batch_size = 1
img_size = 640
img = torch.zeros((batch_size, 3, 384, img_size), device=device, dtype=torch.float32)

torch.onnx.export(
    model,
    img,
    f = OUTPUT_PATH,
    input_names = ["input"],
    output_names = ["det_out", "det_out_dim0", "det_out_dim1", "det_out_dim2", "drive_area_seg", "lane_line_seg"],
    dynamo = False # some issues with dynamo with this model
)

/usr/local/lib/python3.8/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [4]:
# check model
import onnx
model = onnx.load(OUTPUT_PATH)
onnx.checker.check_model(model)